In [0]:
import json

with open("/Workspace/Users/harshamhegde7777@gmail.com/API_Data/key.json", "r") as f:
    config = json.load(f)

api_key = config["api_key"]

# print(config)
print(api_key)

In [0]:
import requests
import json

cities = [
    "Bangalore", "Mumbai", "Delhi", "Hyderabad",
    "Chennai", "Kolkata", "Pune", "Ahmedabad",
    "Jaipur", "Lucknow"
]

import time

data_list = []

for i in range(10):  # run 10 times
    for city in cities:
        url = f"https://api.openweathermap.org/data/2.5/weather?q={city}&appid={api_key}&units=metric"
        
        response = requests.get(url)
        data = response.json()
        
        data_list.append(json.dumps(data))
    
    # time.sleep(60)  # wait 1 minute



In [0]:

df_landing = spark.createDataFrame([(d,) for d in data_list], ["raw_data"])

df_landing.show(truncate=False)

df_landing.printSchema()

df_landing.summary().show()

# spark.read.format("delta") \
#     .option("versionAsOf", 0) \
#     .load("/Volumes/workspace/default/api/landing/")


dbutils.fs.rm("/Volumes/workspace/default/api/landing/", True)

df_landing.write.mode("append").format("delta").save("/Volumes/workspace/default/api/landing/")

In [0]:
df_raw= spark.read.format("delta").load("/Volumes/workspace/default/api/landing/")
df_raw.show()


In [0]:
import json

for row in df_raw.take(20):
    print(json.dumps(json.loads(row["raw_data"]), indent=2))

In [0]:
from pyspark.sql.types import *

schema = StructType([
    StructField("name", StringType()),
    StructField("dt", LongType()),
    StructField("main", StructType([
        StructField("temp", DoubleType()),
        StructField("humidity", IntegerType())
    ])),
    StructField("weather", ArrayType(StructType([
        StructField("description", StringType())
    ])))
])

from pyspark.sql.functions import from_json, col

df_parsed = df_raw.withColumn(
    "json_data",
    from_json(col("raw_data"), schema)
)

df_parsed.printSchema()

df_clean = df_parsed.select(
    col("json_data.name").alias("city"),
    col("json_data.main.temp").alias("temperature"),
    col("json_data.main.humidity").alias("humidity"),
    col("json_data.weather")[0]["description"].alias("weather")
)

display(df_clean)

In [0]:
from pyspark.sql.functions import col, from_unixtime, current_timestamp

from pyspark.sql.functions import col, from_unixtime, current_timestamp

df_silver = df_parsed.select(
    col("json_data.name").alias("city"),
    col("json_data.main.temp").alias("temperature"),
    col("json_data.main.humidity").alias("humidity"),
    col("json_data.weather")[0]["description"].alias("weather"),
    col("json_data.dt").alias("timestamp")
)

df_silver = df_silver.withColumn("event_time", from_unixtime("timestamp")) \
                     .withColumn("ingestion_time", current_timestamp())


df_silver = df_silver.dropDuplicates(["city", "timestamp"])


display(df_silver)

# df_silver.write \
#     .mode("append") \
#     .format("delta") \
#     .save("/Volumes/workspace/default/api/processed/")

Data Quality Checks


In [0]:
df_silver.filter(col("temperature").isNull()).count()
df_silver.filter(col("city").isNull()).count()

In [0]:
dbutils.fs.rm("/Volumes/workspace/default/api/processed/", True)

from pyspark.sql.functions import to_date

df_silver = df_silver.withColumn("date", to_date("event_time"))

df_silver.write \
    .option("mergeSchema", "true") \
    .mode("append") \
    .format("delta") \
    .save("/Volumes/workspace/default/api/processed/")

Creating Delta table on Silver layer

In [0]:
# spark.sql("""
# CREATE TABLE IF NOT EXISTS weather_silver
# USING DELTA
# LOCATION '/Volumes/workspace/default/api/processed/'
# """)

# # spark.sql("""SELECT * FROM weather_silver VERSION AS OF 0;""")
          


Updating Delta table

In [0]:
%sql
-- UPDATE weather_silver
-- SET temperature = temperature + 1
-- WHERE city = 'Bangalore';


Optimization using Z order 

In [0]:
%sql
-- OPTIMIZE weather_silver
-- ZORDER BY (city);


-- DESCRIBE HISTORY weather_silver;

In [0]:
df_processed = spark.read.format("delta") \
    .load("/Volumes/workspace/default/api/processed/")

display(df_processed)

In [0]:
# ✅ 1. Avg Temperature & Humidity per City

from pyspark.sql.functions import avg

df_gold = df_processed.groupBy("city").agg(
    avg("temperature").alias("avg_temperature"),
    avg("humidity").alias("avg_humidity")
)

display(df_gold)

In [0]:
# ✅ 2. Latest Weather per City
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col

window = Window.partitionBy("city").orderBy(col("event_time").desc())

df_latest = df_processed.withColumn("rank", row_number().over(window)) \
                        .filter("rank = 1") \
                        .drop("rank")

display(df_latest)

In [0]:
df_gold.write \
    .mode("overwrite") \
    .format("delta") \
    .save("/Volumes/workspace/default/api/curated/weather_agg")

df_latest.write \
    .mode("overwrite") \
    .format("delta") \
    .save("/Volumes/workspace/default/api/curated/weather_latest")

df_gold.show()
df_latest.show()

Creating delta table on gold layer

In [0]:
# spark.sql("""
# CREATE TABLE IF NOT EXISTS weather_gold
# USING DELTA
# LOCATION '/Volumes/workspace/default/api/curated/weather_agg'
# """)

In [0]:
df_gold.write \
    .mode("overwrite") \
    .saveAsTable("weather_gold_agg")

df_latest.write \
    .mode("overwrite") \
    .saveAsTable("weather_gold_latest")

In [0]:
%sql
DESCRIBE HISTORY weather_gold_agg;

In [0]:
%sql
SELECT * FROM weather_gold_agg VERSION AS OF 0;

In [0]:
%sql
SELECT * FROM weather_gold_agg;